# Step 5 — Model Training & Evaluation (GroupKFold, leakage-free)

## Objective
Train and compare simple models (KNN, Decision Tree, Random Forest, SVM) to predict **rapid vs slow progression** for:
- **3 months (90±7)**
- **6 months (180±7)**

Evaluation is *subject-wise* (GroupKFold) to ensure the same patient does not appear in both train and test.

## Inputs
- `01_data/processed/dataset_3m_v1.csv`
- `01_data/processed/dataset_6m_v1.csv`
*(datasets with features @t0 + slope per horizon)*

## Outputs
- `04_outputs/tables/step5_model_results_v2_thresholded.csv`
- `04_outputs/figures/step5_pr_auc_3m*.png`
- `04_outputs/figures/step5_pr_auc_6m*.png`
- `04_outputs/figures/step5_pr_auc_3m_vs_6m*.png`
- `04_outputs/figures/step5_f2_*.png`
- `04_outputs/figures/step5_fp_fn_*_common.png`

## Main parameters
- Split: **GroupKFold (by patient)**
- Positive class: **rapid = top 30% worst slopes** *(defined within each fold's training set)*
- Threshold: **optimised per fold to maximise F2** *(on training data)*

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Leakage-free note:</b> labels (rapid/slow) and decision threshold are estimated only on each fold's training set. The test set is used exclusively for evaluation.
</div>


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, f1_score, fbeta_score,
    confusion_matrix
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [ ]:
RAW = os.path.join("..", "01_data")
INTERIM = os.path.join("..", "01_data", "interim")
PROCESSED = os.path.join("..", "01_data", "processed")
OUT_TABLES = os.path.join("..", "04_outputs", "tables")

os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(OUT_TABLES, exist_ok=True)

X_path = os.path.join(PROCESSED, "features_baseline_v1.csv")
T_path = os.path.join(INTERIM, "baseline_with_targets_step3_nolabels.csv")

Xdf = pd.read_csv(X_path)
Tdf = pd.read_csv(T_path)

print("Xdf:", Xdf.shape)
print("Tdf:", Tdf.shape)


## 1) Prepare final datasets (3m and 6m)

In this section:
- we load the feature tables (baseline/vitals/respiratory)
- join with slopes per horizon
- ensure **1 row per patient**
- produce two final datasets:
  - `dataset_3m_v1` (90d slope available)
  - `dataset_6m_v1` (180d slope available)

<b>Practical goal:</b> ensure that training uses only features @t0 and the target comes from the future period (t0→t0+H).


In [ ]:
df = Xdf.merge(
    Tdf[["subject_id", "slope_90d_per_30d", "slope_180d_per_30d"]],
    on="subject_id", how="inner"
)

df3 = df[df["slope_90d_per_30d"].notna()].copy()
df6 = df[df["slope_180d_per_30d"].notna()].copy()

print("Dataset 3m:", df3.shape, "| N subjects:", df3["subject_id"].nunique())
print("Dataset 6m:", df6.shape, "| N subjects:", df6["subject_id"].nunique())


## 1.1) Save training-ready datasets (reproducibility)

We save "frozen" versions of the 3m and 6m datasets to:
- reproduce results (without reprocessing everything)
- allow column/row auditing
- simplify Step 6 (XAI)

<b>Rule:</b> these files should not be changed manually; any modifications must come from the pipeline (Step 4).


In [ ]:
df3.to_csv(os.path.join(PROCESSED, "dataset_3m_v1.csv"), index=False)
df6.to_csv(os.path.join(PROCESSED, "dataset_6m_v1.csv"), index=False)


## 2) Define X (features) and y-base (slope)

Here we choose:
- **X:** predictor columns (baseline/t0 only)
- **slope:** continuous variable used to create the binary rapid/slow label per fold
- **groups:** patient identifier (for GroupKFold)

<b>Note:</b> we do not fix a global "binary y" here. The rapid/slow label is created within each fold from the training slope (top 30%).


In [ ]:
DROP_ALWAYS = {
    "subject_id",
    "slope_90d_per_30d", "slope_180d_per_30d",
    "slope_90d_per_day", "slope_180d_per_day",
    "npoints_90d_window", "npoints_180d_window",
    "has_endband_90d", "has_endband_180d",
    "pref_ok_90d_min3", "pref_ok_180d_min3",
}

def split_X_y(df_in, slope_col):
    cols = [c for c in df_in.columns if c not in DROP_ALWAYS]
    X = df_in[cols].copy()
    slope = df_in[slope_col].copy()
    groups = df_in["subject_id"].copy()
    return X, slope, groups

X3, slope3, g3 = split_X_y(df3, "slope_90d_per_30d")
X6, slope6, g6 = split_X_y(df6, "slope_180d_per_30d")

print("X3:", X3.shape, "| X6:", X6.shape)


## 3) Preprocessing (numeric + categorical)

Goal: prepare data consistently for different models.

- Numeric: imputation (median) + (optional) scaling
- Categorical: imputation (most frequent) + one-hot encoding
- Scale-sensitive models (KNN, SVM): use pipeline with scaler
- Tree-based models (DecisionTree, RandomForest): do not need scaling

<b>Why this matters:</b> prevents failures from strings ("Male") and reduces missingness bias.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# detect columns by type
num_cols_3 = X3.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_3 = [c for c in X3.columns if c not in num_cols_3]

num_cols_6 = X6.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_6 = [c for c in X6.columns if c not in num_cols_6]

print("3m -> num:", len(num_cols_3), "cat:", len(cat_cols_3))
print("6m -> num:", len(num_cols_6), "cat:", len(cat_cols_6))
print("Example cat 3m:", cat_cols_3[:10])

num_preprocess_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

num_preprocess_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

cat_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


## 3.1) ColumnTransformer (reproducible pipeline)

We use ColumnTransformer to apply different transformations per variable type:
- numeric columns → imputation/scaling
- categorical columns → imputation/one-hot

<b>Advantage:</b> the same pipeline serves training, testing, and final model export.


In [ ]:
from sklearn.compose import ColumnTransformer

# Preprocessadores para modelos que precisam de scaling (KNN, SVM)
preproc_scaled_3 = ColumnTransformer(
    transformers=[
        ("num", num_preprocess_scaled, num_cols_3),
        ("cat", cat_preprocess, cat_cols_3),
    ],
    remainder="drop"
)

preproc_scaled_6 = ColumnTransformer(
    transformers=[
        ("num", num_preprocess_scaled, num_cols_6),
        ("cat", cat_preprocess, cat_cols_6),
    ],
    remainder="drop"
)

# Preprocessadores para árvores (DT, RF) (sem scaler)
preproc_tree_3 = ColumnTransformer(
    transformers=[
        ("num", num_preprocess_tree, num_cols_3),
        ("cat", cat_preprocess, cat_cols_3),
    ],
    remainder="drop"
)

preproc_tree_6 = ColumnTransformer(
    transformers=[
        ("num", num_preprocess_tree, num_cols_6),
        ("cat", cat_preprocess, cat_cols_6),
    ],
    remainder="drop"
)


## 4) Helper function: threshold selection (max F2)

Instead of using a fixed threshold (0.5), we select a threshold that maximises **F2** on the fold's training set.

<b>Motivation:</b> the problem is imbalanced and, clinically, we penalise missing a "rapid" case (FN) more than generating false alarms (FP).


In [ ]:
import numpy as np
from sklearn.metrics import fbeta_score

def pick_threshold_max_fbeta(y_true, proba, beta=2, grid=None):
    """
    Choose the threshold that maximises F-beta on (y_true, proba).
    grid: list/array of thresholds to test.
    """
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)

    best_t = 0.5
    best_s = -1.0

    for t in grid:
        yhat = (proba >= t).astype(int)
        s = fbeta_score(y_true, yhat, beta=beta, zero_division=0)
        if s > best_s:
            best_s = s
            best_t = float(t)

    return best_t, float(best_s)


## 5) Leakage-free evaluation with GroupKFold (labels and threshold per fold)

For each model and each fold:
1) subject-wise split (GroupKFold)
2) on training set: compute `thr_label` = 30th percentile of slope (worst declines)
3) create binary y (rapid = slope ≤ thr_label)
4) train model
5) on training set: choose `thr_decision` that maximises F2
6) on test set: evaluate PR-AUC (threshold-free) and F2/F1 + FP/FN (thresholded)

<b>Output:</b> mean and standard deviation metrics per model and per horizon.


In [ ]:
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, confusion_matrix

def evaluate_models(X, slope, groups, horizon_name, preproc_scaled, preproc_tree,
                    top_frac=0.30, n_splits=5, random_state=42,
                    beta=2, thr_grid=None):

    gkf = GroupKFold(n_splits=n_splits)

    models = [
        ("KNN", Pipeline(steps=[
            ("prep", preproc_scaled),
            ("clf", KNeighborsClassifier(n_neighbors=15))
        ])),
        ("DecisionTree", Pipeline(steps=[
            ("prep", preproc_tree),
            ("clf", DecisionTreeClassifier(max_depth=5, random_state=random_state))
        ])),
        ("RandomForest", Pipeline(steps=[
            ("prep", preproc_tree),
            ("clf", RandomForestClassifier(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=2,
                random_state=random_state,
                n_jobs=-1
            ))
        ])),
        ("SVM", Pipeline(steps=[
            ("prep", preproc_scaled),
            ("clf", SVC(
                C=1.0, kernel="rbf",
                probability=True,
                class_weight="balanced",
                random_state=random_state
            ))
        ])),
    ]

    rows = []
    for name, pipe in models:
        pr_list, f1_list, f2_list = [], [], []
        tn_list, fp_list, fn_list, tp_list = [], [], [], []
        thr_list = []

        for fold, (tr, te) in enumerate(gkf.split(X, groups=groups), start=1):
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            slope_tr, slope_te = slope.iloc[tr], slope.iloc[te]

            # 1) label threshold (top 30% most negative) computed on training only
            thr_label = np.nanquantile(slope_tr.dropna(), top_frac)
            ytr = (slope_tr <= thr_label).astype(int)
            yte = (slope_te <= thr_label).astype(int)

            # 2) train
            pipe.fit(Xtr, ytr)

            # 3) choose decision threshold on training (max F-beta)
            proba_tr = pipe.predict_proba(Xtr)[:, 1]
            thr_decision, _ = pick_threshold_max_fbeta(ytr.to_numpy(), proba_tr, beta=beta, grid=thr_grid)
            thr_list.append(thr_decision)

            # 4) evaluate on test with that threshold
            proba_te = pipe.predict_proba(Xte)[:, 1]
            yhat = (proba_te >= thr_decision).astype(int)

            pr = average_precision_score(yte, proba_te)
            f1 = f1_score(yte, yhat, zero_division=0)
            f2 = fbeta_score(yte, yhat, beta=beta, zero_division=0)

            tn, fp, fn, tp = confusion_matrix(yte, yhat, labels=[0, 1]).ravel()

            pr_list.append(pr); f1_list.append(f1); f2_list.append(f2)
            tn_list.append(tn); fp_list.append(fp); fn_list.append(fn); tp_list.append(tp)

        rows.append({
            "horizon": horizon_name,
            "model": name,

            "PR_AUC_mean": float(np.mean(pr_list)),
            "PR_AUC_std": float(np.std(pr_list)),

            "F1_mean": float(np.mean(f1_list)),
            "F1_std": float(np.std(f1_list)),

            f"F{beta}_mean": float(np.mean(f2_list)),
            f"F{beta}_std": float(np.std(f2_list)),

            "TN_mean": float(np.mean(tn_list)),
            "FP_mean": float(np.mean(fp_list)),
            "FN_mean": float(np.mean(fn_list)),
            "TP_mean": float(np.mean(tp_list)),

            "thr_decision_mean": float(np.mean(thr_list)),
            "thr_decision_std": float(np.std(thr_list)),
        })

    return pd.DataFrame(rows)


## 6) Run training/evaluation (3m and 6m) and save results

In this section:
- we run the evaluation for 3m and 6m
- combine into a single table
- save as a thesis-ready CSV with:
  - PR-AUC_mean/std
  - F1_mean/std
  - F2_mean/std
  - FP/FN/TP/TN (means)
  - mean decision threshold (thr_decision_mean/std)


In [ ]:
res3 = evaluate_models(
    X3, slope3, g3,
    horizon_name="3m",
    preproc_scaled=preproc_scaled_3,
    preproc_tree=preproc_tree_3,
    top_frac=0.30,
    n_splits=5,
    beta=2
)

res6 = evaluate_models(
    X6, slope6, g6,
    horizon_name="6m",
    preproc_scaled=preproc_scaled_6,
    preproc_tree=preproc_tree_6,
    top_frac=0.30,
    n_splits=5,
    beta=2
)

results = pd.concat([res3, res6], ignore_index=True)
results = results.sort_values(["horizon", "PR_AUC_mean"], ascending=[True, False])

out = os.path.join(OUT_TABLES, "step5_model_results_v2_thresholded.csv")
results.to_csv(out, index=False)

print("Saved:", out)
results


## 7) Quick visualisation (sanity check)

Before producing final figures, we make a simple plot to confirm:
- model ordering
- values within plausible ranges
- absence of degenerate behaviour (e.g., F2=0 from always predicting 0)

<b>Note:</b> final figures follow in subsequent sections.


In [ ]:
import matplotlib.pyplot as plt

def barplot_metric(df, horizon, metric):
    sub = df[df["horizon"] == horizon].sort_values(metric, ascending=False)
    plt.figure(figsize=(8,4))
    plt.bar(sub["model"], sub[metric])
    plt.title(f"{metric} per model — {horizon}")
    plt.ylabel(metric)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

barplot_metric(results, "3m", "PR_AUC_mean")
barplot_metric(results, "6m", "PR_AUC_mean")


## 8) Final figures: PR-AUC (common scale + error + baseline)

We generate thesis-ready plots:
- PR-AUC per model (3m and 6m, separate)
- same Y scale for comparability
- error bars (std across folds)
- baseline line = 0.30 (rapid prevalence)

The "+0.XX" annotations represent absolute gain over baseline: PR-AUC − 0.30.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_metric_bars(results_sub, metric_mean, metric_std, title,
                     outfile=None, y_min=None, y_max=None,
                     baseline=None, note=None,
                     bar_color="#50eb79", baseline_color="#0a0807"):

    df = results_sub.copy().sort_values(metric_mean, ascending=False)

    x = np.arange(len(df))
    y = df[metric_mean].to_numpy()
    yerr = df[metric_std].to_numpy()

    plt.figure(figsize=(9,4.5))
    plt.bar(x, y, yerr=yerr, capsize=5, color=bar_color)

    plt.xticks(x, df["model"], rotation=15)
    plt.ylabel(metric_mean.replace("_mean",""))
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)

    # ---- Y limits: auto if None ----
    top = float(np.max(y + yerr))
    bot = float(np.min(y - yerr))

    if y_min is None:
        y_min = max(0.0, bot - 0.05)
    if y_max is None:
        y_max = min(1.0, top + 0.05)

    plt.ylim(y_min, y_max)

    if baseline is not None:
        plt.axhline(baseline, linestyle="--", linewidth=2, color=baseline_color,
                    label=f"baseline={baseline:.2f}")

        # text above bars (use error to avoid collision)
        for i, (val, err) in enumerate(zip(y, yerr)):
            delta = val - baseline
            plt.text(i, val + err + 0.01, f"{delta:+.02f} above baseline",
                     ha="center", va="bottom", fontsize=9)

        plt.legend(loc="lower right")

    if note:
        plt.text(0.01, 0.98, note, transform=plt.gca().transAxes,
                 va="top", ha="left", fontsize=9)

    plt.tight_layout()

    if outfile:
        plt.savefig(outfile, dpi=300)
    plt.show()


In [ ]:
OUT_FIGURES = os.path.join("..", "04_outputs", "figures")
os.makedirs(OUT_FIGURES, exist_ok=True)
res3_sub = results[results["horizon"]=="3m"]
res6_sub = results[results["horizon"]=="6m"]

plot_metric_bars(
    res3_sub, "PR_AUC_mean", "PR_AUC_std",
    "PR-AUC per model — 3 months (GroupKFold)",
    outfile=os.path.join(OUT_FIGURES, "step5_pr_auc_3m_pretty.png"),
    baseline=0.30,
    note="baseline = prevalence (top 30%)"
)

plot_metric_bars(
    res6_sub, "PR_AUC_mean", "PR_AUC_std",
    "PR-AUC per model — 6 months (GroupKFold)",
    outfile=os.path.join(OUT_FIGURES, "step5_pr_auc_6m_pretty.png"),
    baseline=0.30,
    note="baseline = prevalence (top 30%)"
)


## 8.1) PR-AUC 3m vs 6m (combined figure)

In this figure we place 3m and 6m side by side per model to:
- directly compare the impact of the temporal horizon
- check whether the model hierarchy is stable
- keep baseline and uncertainty (std)

<b>Note:</b> annotations per bar show gain above baseline.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def pr_auc_grouped_with_gain(results, outfile=None, baseline=0.30,
                             color_3m="#FF5F5D", color_6m="#3F7C85",
                             y_min=0.25, y_max=0.50):

    r3 = results[results["horizon"] == "3m"].set_index("model")
    r6 = results[results["horizon"] == "6m"].set_index("model")

    # fixed order by 6m
    order = r6["PR_AUC_mean"].sort_values(ascending=False).index.tolist()
    r3 = r3.loc[order]
    r6 = r6.loc[order]

    x = np.arange(len(order))
    w = 0.28

    plt.figure(figsize=(12, 5))

    bars3 = plt.bar(
        x - w/2, r3["PR_AUC_mean"], yerr=r3["PR_AUC_std"],
        capsize=5, width=w, label="3 months", color=color_3m
    )
    bars6 = plt.bar(
        x + w/2, r6["PR_AUC_mean"], yerr=r6["PR_AUC_std"],
        capsize=5, width=w, label="6 months", color=color_6m
    )

    plt.axhline(baseline, linestyle="--", linewidth=2, color="#222222",
                label=f"baseline={baseline:.2f}")

    plt.xticks(x, order, rotation=15)
    plt.ylabel("PR-AUC")
    plt.title("PR-AUC per model — 3m vs 6m (GroupKFold)")
    plt.ylim(y_min, y_max)
    plt.grid(True, axis="y", alpha=0.25)

    # ---- gains (above baseline) on top of each bar ----
    def annotate_gain(bar_container, means):
        for rect, val in zip(bar_container, means):
            gain = val - baseline
            y_text = rect.get_height() + 0.006
            plt.text(
                rect.get_x() + rect.get_width()/2,
                y_text,
                f"{gain:+.02f}",
                ha="left", va="bottom", fontsize=9
            )

    annotate_gain(bars3, r3["PR_AUC_mean"].to_numpy())
    annotate_gain(bars6, r6["PR_AUC_mean"].to_numpy())

    # fixed note (to avoid repeating "above baseline" 8 times)
    plt.text(
        0.01, 0.98,
        "notes: Gain above baseline (PR-AUC - 0.30)",
        transform=plt.gca().transAxes,
        ha="left", va="top", fontsize=9
    )

    plt.legend(loc="upper right")
    plt.tight_layout()

    if outfile:
        plt.savefig(outfile, dpi=300)
    plt.show()

pr_auc_grouped_with_gain(
    results,
    outfile=os.path.join(OUT_FIGURES, "step5_pr_auc_3m_vs_6m_gain.png"),
    baseline=0.30
)


## 9) Final figures: F2 and FP vs FN trade-off

- F2 per model (3m and 6m), with threshold optimised per fold.
- FP vs FN plot (common scale), to make operational cost explicit:
  - FN: missed rapid progressors
  - FP: false alarms

<b>Expected interpretation:</b> more "aggressive" models increase F2 by reducing FN, but at the cost of higher FP.


In [ ]:
res3_sub = results[results["horizon"]=="3m"]
res6_sub = results[results["horizon"]=="6m"]

plot_metric_bars(
    res3_sub, "F2_mean", "F2_std",
    "F2 per model — 3 months (threshold optimised per fold)",
    outfile=os.path.join(OUT_FIGURES, "step5_f2_3m.png"),
    baseline=None,
    note="threshold chosen on training (max F2)"
)

plot_metric_bars(
    res6_sub, "F2_mean", "F2_std",
    "F2 per model — 6 months (threshold optimised per fold)",
    outfile=os.path.join(OUT_FIGURES, "step5_f2_6m.png"),
    baseline=None,
    note="threshold chosen on training (max F2)"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_fp_fn(results_sub, title, outfile=None,
               ylim_top=None, base_color="#FF5F5D",
               show_values=True):

    order = ["SVM", "RandomForest", "DecisionTree", "KNN"]
    df = results_sub.copy().set_index("model").loc[order]

    x = np.arange(len(df))
    w = 0.40

    color_fn = base_color
    color_fp = base_color

    plt.figure(figsize=(9.5, 4.8))

    b_fn = plt.bar(
        x - w/2, df["FN_mean"], width=w,
        label="FN (missed rapids)",
        color=color_fn, alpha=0.90, edgecolor="black", linewidth=0.6
    )
    b_fp = plt.bar(
        x + w/2, df["FP_mean"], width=w,
        label="FP (false alarms)",
        color=color_fp, alpha=0.45, edgecolor="black", linewidth=0.6, hatch="//"
    )

    plt.xticks(x, df.index, rotation=15)
    plt.ylabel("mean per fold")
    plt.title(title)

    # common scale (if provided)
    if ylim_top is not None:
        plt.ylim(0, ylim_top)

    plt.grid(True, axis="y", alpha=0.25)

    # values on top of bars (optional but helpful)
    if show_values:
        for bars in (b_fn, b_fp):
            for r in bars:
                h = r.get_height()
                plt.text(
                    r.get_x() + r.get_width()/2,
                    h + (ylim_top*0.015 if ylim_top else 2),
                    f"{h:.0f}",
                    ha="center", va="bottom", fontsize=9
                )

    plt.legend(loc="upper left")
    plt.tight_layout()

    if outfile:
        plt.savefig(outfile, dpi=300)
    plt.show()


# --- compute global maximum for common scale ---
def compute_common_ylim(results_3m, results_6m, pad=1.10):
    vals = []
    for r in (results_3m, results_6m):
        vals.extend(r["FN_mean"].tolist())
        vals.extend(r["FP_mean"].tolist())
    m = max(vals) if vals else 1
    return m * pad


ylim_common = compute_common_ylim(res3_sub, res6_sub, pad=1.10)

plot_fp_fn(
    res3_sub,
    "FP vs FN trade-off — 3 months",
    outfile=os.path.join(OUT_FIGURES, "step5_fp_fn_3m_common.png"),
    ylim_top=ylim_common,
    base_color="#FF5F5D"
)

plot_fp_fn(
    res6_sub,
    "FP vs FN trade-off — 6 months",
    outfile=os.path.join(OUT_FIGURES, "step5_fp_fn_6m_common.png"),
    ylim_top=ylim_common,
    base_color="#3F7C85"
)


## Takeaways (for the thesis)

- PR-AUC compares models robustly (threshold-independent).
- Threshold optimised per fold (max F2) defines an operating point aligned with high sensitivity.
- The FP/FN trade-off clearly distinguishes aggressive vs conservative models.
- Results are leakage-free: subject-wise splits + labels/threshold defined on training data.


# Step 5.1 — Ablation Study (SVM vs RandomForest) — Feature Blocks

## Objective
Measure **where the predictive signal comes from** by comparing feature blocks, maintaining:
- the same leakage-free protocol (GroupKFold subject-wise)
- rapid/slow labels defined per fold (top 30% worst slopes on training)
- threshold optimised per fold (max F2 on training)

## Models (fixed in this step)
- SVM
- RandomForest

## Feature blocks evaluated
A) Baseline-only  
B) Baseline + Vitals  
C) Baseline + FVC  
D) Baseline + Vitals + FVC

## Expected results
- Table with PR-AUC and F2 (mean ± std) per horizon and block
- Main figure: **dumbbell plot** of incremental gain per block (clearer than bars)
- (Optional) heatmap for global overview


In [ ]:
# --- Step 5.1: define feature blocks (heuristic by column name) ---

def infer_feature_blocks(X):
    cols = list(X.columns)

    # detect blocks by common patterns
    vitals_cols = [c for c in cols if ("vital" in c.lower()) or (c.lower().endswith("_t0") and any(k in c.lower() for k in ["weight","height","bmi","bp","pulse","temp","resp","hr"]))]
    fvc_cols    = [c for c in cols if "fvc" in c.lower()]
    
    # baseline = everything that is not vitals or fvc
    baseline_cols = [c for c in cols if (c not in vitals_cols) and (c not in fvc_cols)]

    blocks = {
        "A_baseline": baseline_cols,
        "B_baseline+vitals": sorted(set(baseline_cols + vitals_cols)),
        "C_baseline+fvc": sorted(set(baseline_cols + fvc_cols)),
        "D_baseline+vitals+fvc": sorted(set(baseline_cols + vitals_cols + fvc_cols)),
    }
    # cleanup: remove duplicates and ensure not empty
    blocks = {k: [c for c in v if c in cols] for k, v in blocks.items()}
    return blocks, vitals_cols, fvc_cols, baseline_cols

blocks3, vitals3, fvc3, base3 = infer_feature_blocks(X3)
blocks6, vitals6, fvc6, base6 = infer_feature_blocks(X6)

print("3m | baseline:", len(base3), "vitals:", len(vitals3), "fvc:", len(fvc3))
print("6m | baseline:", len(base6), "vitals:", len(vitals6), "fvc:", len(fvc6))

# sanity check: ensure blocks have columns
for name, cols_ in blocks3.items():
    print("3m", name, "->", len(cols_))
for name, cols_ in blocks6.items():
    print("6m", name, "->", len(cols_))


## 5.1.1) Run ablation (SVM and RandomForest) per block

For each horizon (3m and 6m) we evaluate the 4 blocks A–D.  
The protocol remains exactly the same as Step 5:
- rapid/slow labels defined per fold (top 30% worst slopes on training)
- decision threshold chosen per fold to maximise F2 on training
- evaluation on test per fold, with mean ± std


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_preprocessors_for_X(X):
    # colunas numéricas/categóricas a partir do dataframe atual
    num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    num_pipe_scaled = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    num_pipe_tree = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preproc_scaled = ColumnTransformer(
        transformers=[
            ("num", num_pipe_scaled, num_cols),
            ("cat", cat_pipe, cat_cols)
        ],
        remainder="drop"
    )

    preproc_tree = ColumnTransformer(
        transformers=[
            ("num", num_pipe_tree, num_cols),
            ("cat", cat_pipe, cat_cols)
        ],
        remainder="drop"
    )

    return preproc_scaled, preproc_tree, num_cols, cat_cols


In [ ]:
# --- Step 5.1: correr ablation por blocos ---

ABL_MODELS = ["SVM", "RandomForest"]

def run_ablation(X, slope, groups, blocks, horizon_name,
                 top_frac=0.30, n_splits=5):

    rows = []
    for block_name, cols_ in blocks.items():
        Xb = X[cols_].copy()

        preproc_scaled_b, preproc_tree_b, num_cols, cat_cols = build_preprocessors_for_X(Xb)

        res = evaluate_models(
            Xb, slope, groups,
            horizon_name=horizon_name,
            preproc_scaled=preproc_scaled_b,
            preproc_tree=preproc_tree_b,
            top_frac=top_frac,
            n_splits=n_splits
        )

        res = res[res["model"].isin(ABL_MODELS)].copy()
        res["feature_block"] = block_name
        res["n_features"] = len(cols_)
        res["n_num"] = len(num_cols)
        res["n_cat"] = len(cat_cols)
        rows.append(res)

    return pd.concat(rows, ignore_index=True)




abl3 = run_ablation(X3, slope3, g3, blocks3, horizon_name="3m", top_frac=0.30, n_splits=5)
abl6 = run_ablation(X6, slope6, g6, blocks6, horizon_name="6m", top_frac=0.30, n_splits=5)

ablation_results = pd.concat([abl3, abl6], ignore_index=True)
ablation_results[["horizon","feature_block","model","n_features","PR_AUC_mean","F2_mean"]].head(12)




## 5.1.2) Save results (thesis-ready)

We save a single table with metrics per:
- horizon (3m/6m)
- model (SVM/RF)
- feature block (A–D)


In [ ]:
out_csv = os.path.join(OUT_TABLES, "step5_1_ablation_svm_rf_fvc.csv")
ablation_results.to_csv(out_csv, index=False)
print("Saved:", out_csv)


## 5.1.3) Main figure: evolution by block (dumbbell plot)

The dumbbell plot connects performance across blocks, showing:
- incremental gain (baseline → +vitals → +FVC → +both)
- differences between SVM and RF
- clean comparison of 3m vs 6m


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BLOCK_ORDER = ["A_baseline", "B_baseline+vitals", "C_baseline+fvc", "D_baseline+vitals+fvc"]

def dumbbell_plot(df, metric_mean, metric_std, horizon, outfile=None):
    plt.figure(figsize=(10, 4.8))

    for model_name in ["SVM", "RandomForest"]:
        d = df[(df["horizon"] == horizon) & (df["model"] == model_name)].copy()
        d["feature_block"] = pd.Categorical(d["feature_block"], categories=BLOCK_ORDER, ordered=True)
        d = d.sort_values("feature_block")

        y = np.arange(len(BLOCK_ORDER))
        x = d.set_index("feature_block").loc[BLOCK_ORDER, metric_mean].to_numpy()
        e = d.set_index("feature_block").loc[BLOCK_ORDER, metric_std].to_numpy()

        # lines between points
        plt.plot(x, y, marker="o", linewidth=2, label=model_name)
        # error
        plt.errorbar(x, y, xerr=e, fmt="none", capsize=4, linewidth=1)

    plt.yticks(np.arange(len(BLOCK_ORDER)), BLOCK_ORDER)
    plt.xlabel(metric_mean.replace("_mean",""))
    plt.title(f"{metric_mean.replace('_mean','')} — Ablation by block ({horizon})")
    plt.grid(True, axis="x", alpha=0.25)
    plt.legend(loc="lower right")
    plt.tight_layout()

    if outfile:
        plt.savefig(outfile, dpi=300)
        print("Saved:", outfile)
    plt.show()

# PR-AUC
dumbbell_plot(ablation_results, "PR_AUC_mean", "PR_AUC_std", "3m",
              outfile=os.path.join(OUT_FIGURES, "step5_1_ablation_dumbbell_pr_auc_3m.png"))
dumbbell_plot(ablation_results, "PR_AUC_mean", "PR_AUC_std", "6m",
              outfile=os.path.join(OUT_FIGURES, "step5_1_ablation_dumbbell_pr_auc_6m.png"))


In [ ]:
# F2
dumbbell_plot(ablation_results, "F2_mean", "F2_std", "3m",
              outfile=os.path.join(OUT_FIGURES, "step5_1_ablation_dumbbell_f2_3m.png"))
dumbbell_plot(ablation_results, "F2_mean", "F2_std", "6m",
              outfile=os.path.join(OUT_FIGURES, "step5_1_ablation_dumbbell_f2_6m.png"))


In [ ]:
import matplotlib.pyplot as plt

def heatmap_ablation(df, horizon, metric="PR_AUC_mean", outfile=None):
    d = df[df["horizon"] == horizon].copy()
    d = d[d["model"].isin(["SVM","RandomForest"])]
    d["feature_block"] = pd.Categorical(d["feature_block"], categories=BLOCK_ORDER, ordered=True)

    pivot = d.pivot_table(index="model", columns="feature_block", values=metric, aggfunc="mean").loc[["SVM","RandomForest"], BLOCK_ORDER]

    plt.figure(figsize=(9, 2.6))
    plt.imshow(pivot.values, aspect="auto")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=20)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title(f"{metric} — Ablation heatmap ({horizon})")
    plt.colorbar()
    plt.tight_layout()

    if outfile:
        plt.savefig(outfile, dpi=300)
        print("Saved:", outfile)
    plt.show()

heatmap_ablation(ablation_results, "3m", metric="PR_AUC_mean",
                 outfile=os.path.join(OUT_FIGURES, "step5_1_ablation_heatmap_pr_auc_3m.png"))
heatmap_ablation(ablation_results, "6m", metric="PR_AUC_mean",
                 outfile=os.path.join(OUT_FIGURES, "step5_1_ablation_heatmap_pr_auc_6m.png"))


# Step 5.2 — Extra: Logistic Regression (interpretable baseline)

## Objective
Add Logistic Regression as an interpretable baseline while maintaining the same protocol:
- GroupKFold subject-wise
- rapid = top 30% worst slopes (per fold)
- threshold optimised per fold to maximise F2

This block does not alter previous results (KNN/DT/RF/SVM); it generates a separate output.


In [ ]:
from sklearn.linear_model import LogisticRegression

def evaluate_logreg(X, slope, groups, horizon_name,
                    preproc_scaled,
                    top_frac=0.30, n_splits=5, beta=2, thr_grid=None, random_state=42):

    gkf = GroupKFold(n_splits=n_splits)

    pipe = Pipeline(steps=[
        ("prep", preproc_scaled),
        ("clf", LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            class_weight="balanced",
            random_state=random_state
        ))
    ])

    pr_list, f1_list, f2_list = [], [], []
    tn_list, fp_list, fn_list, tp_list = [], [], [], []
    thr_list = []

    for fold, (tr, te) in enumerate(gkf.split(X, groups=groups), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        slope_tr, slope_te = slope.iloc[tr], slope.iloc[te]

        thr_label = np.nanquantile(slope_tr.dropna(), top_frac)
        ytr = (slope_tr <= thr_label).astype(int)
        yte = (slope_te <= thr_label).astype(int)

        pipe.fit(Xtr, ytr)

        proba_tr = pipe.predict_proba(Xtr)[:, 1]
        thr_decision, _ = pick_threshold_max_fbeta(ytr.to_numpy(), proba_tr, beta=beta, grid=thr_grid)
        thr_list.append(thr_decision)

        proba_te = pipe.predict_proba(Xte)[:, 1]
        yhat = (proba_te >= thr_decision).astype(int)

        pr_list.append(average_precision_score(yte, proba_te))
        f1_list.append(f1_score(yte, yhat, zero_division=0))
        f2_list.append(fbeta_score(yte, yhat, beta=beta, zero_division=0))

        tn, fp, fn, tp = confusion_matrix(yte, yhat, labels=[0,1]).ravel()
        tn_list.append(tn); fp_list.append(fp); fn_list.append(fn); tp_list.append(tp)

    return pd.DataFrame([{
        "horizon": horizon_name,
        "model": "LogReg_balanced",
        "PR_AUC_mean": float(np.mean(pr_list)),
        "PR_AUC_std": float(np.std(pr_list)),
        "F1_mean": float(np.mean(f1_list)),
        "F1_std": float(np.std(f1_list)),
        f"F{beta}_mean": float(np.mean(f2_list)),
        f"F{beta}_std": float(np.std(f2_list)),
        "TN_mean": float(np.mean(tn_list)),
        "FP_mean": float(np.mean(fp_list)),
        "FN_mean": float(np.mean(fn_list)),
        "TP_mean": float(np.mean(tp_list)),
        "thr_decision_mean": float(np.mean(thr_list)),
        "thr_decision_std": float(np.std(thr_list)),
    }])
    


In [ ]:
lr3 = evaluate_logreg(X3, slope3, g3, "3m", preproc_scaled_3, top_frac=0.30, n_splits=5, beta=2)
lr6 = evaluate_logreg(X6, slope6, g6, "6m", preproc_scaled_6, top_frac=0.30, n_splits=5, beta=2)

lr_results = pd.concat([lr3, lr6], ignore_index=True)

out = os.path.join(OUT_TABLES, "step5_logreg_results.csv")
lr_results.to_csv(out, index=False)

print("Saved:", out)
lr_results


# Step 5.3 — Extra: Gradient Boosting (LightGBM + XGBoost)

## Objective
Add two gradient-boosting models for comparison with classical models:
- LightGBM
- XGBoost

We maintain the same leakage-free protocol:
- GroupKFold (subject-wise, by patient)
- rapid = top 30% worst slopes (defined per fold on training)
- threshold optimised per fold to maximise F2 (on training)
- evaluation on test with PR-AUC + F1/F2 + FP/FN

## Outputs
- `04_outputs/tables/step5_gbdt_results.csv`
- (optional) PR-AUC/F2 plots with these models


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, f1_score, fbeta_score, confusion_matrix
)

try:
    from xgboost import XGBClassifier
except Exception as e:
    XGBClassifier = None
    print("[WARNING] xgboost is not available:", e)

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    LGBMClassifier = None
    print("[WARNING] lightgbm is not available:", e)


## 5.3.1) Evaluation function (same protocol as Step 5)

This function replicates the Step 5 logic:
- in each fold defines rapid by the top 30% of training slopes
- trains the model
- chooses the threshold on training that maximises F2
- evaluates on test (PR-AUC, F1, F2, FP/FN)


In [ ]:
def evaluate_gbdt_models(
    X, slope, groups,
    horizon_name,
    preproc_tree,
    top_frac=0.30,
    n_splits=5,
    beta=2,
    thr_grid=None,
    random_state=42
):
    """
    Evaluate LightGBM and XGBoost with the same Step 5 protocol.
    Returns a DataFrame with aggregated metrics (mean ± std).
    """

    if (XGBClassifier is None) and (LGBMClassifier is None):
        raise RuntimeError("Neither xgboost nor lightgbm are available in this environment.")

    gkf = GroupKFold(n_splits=n_splits)

    # models (reasonable defaults to start)
    models = []

    if LGBMClassifier is not None:
        models.append((
            "LightGBM",
            LGBMClassifier(
                n_estimators=400,
                learning_rate=0.05,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=random_state
            )
        ))

    if XGBClassifier is not None:
        models.append((
            "XGBoost",
            XGBClassifier(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_state,
                n_jobs=-1
            )
        ))

    rows = []

    for model_name, clf in models:
        pr_list, f1_list, f2_list = [], [], []
        tn_list, fp_list, fn_list, tp_list = [], [], [], []
        thr_list = []

        for fold, (tr, te) in enumerate(gkf.split(X, groups=groups), start=1):
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            slope_tr, slope_te = slope.iloc[tr], slope.iloc[te]

            # label fold-wise (rapid = worst 30% slopes on training)
            thr_label = float(np.nanquantile(slope_tr.dropna(), top_frac))
            ytr = (slope_tr <= thr_label).astype(int)
            yte = (slope_te <= thr_label).astype(int)

            pipe = Pipeline([
                ("prep", preproc_tree),
                ("clf", clf)
            ])

            pipe.fit(Xtr, ytr)

            # choose threshold on training max F2
            proba_tr = pipe.predict_proba(Xtr)[:, 1]
            thr_decision, _ = pick_threshold_max_fbeta(
                ytr.to_numpy(), proba_tr, beta=beta, grid=thr_grid
            )
            thr_list.append(float(thr_decision))

            # evaluate on test
            proba_te = pipe.predict_proba(Xte)[:, 1]
            yhat = (proba_te >= thr_decision).astype(int)

            pr_list.append(average_precision_score(yte, proba_te))
            f1_list.append(f1_score(yte, yhat, zero_division=0))
            f2_list.append(fbeta_score(yte, yhat, beta=beta, zero_division=0))

            tn, fp, fn, tp = confusion_matrix(yte, yhat, labels=[0, 1]).ravel()
            tn_list.append(tn); fp_list.append(fp); fn_list.append(fn); tp_list.append(tp)

        rows.append({
            "horizon": horizon_name,
            "model": model_name,
            "PR_AUC_mean": float(np.mean(pr_list)),
            "PR_AUC_std": float(np.std(pr_list)),
            "F1_mean": float(np.mean(f1_list)),
            "F1_std": float(np.std(f1_list)),
            "F2_mean": float(np.mean(f2_list)),
            "F2_std": float(np.std(f2_list)),
            "TN_mean": float(np.mean(tn_list)),
            "FP_mean": float(np.mean(fp_list)),
            "FN_mean": float(np.mean(fn_list)),
            "TP_mean": float(np.mean(tp_list)),
            "thr_decision_mean": float(np.mean(thr_list)),
            "thr_decision_std": float(np.std(thr_list)),
        })

    return pd.DataFrame(rows)


## 5.3.2) Run (3m and 6m) and save results

We generate a separate CSV so as not to alter the previous Step 5 outputs.


In [ ]:
# 3m and 6m (uses the preprocessors already defined in Step 5)
gbdt3 = evaluate_gbdt_models(
    X3, slope3, g3,
    horizon_name="3m",
    preproc_tree=preproc_tree_3,
    top_frac=0.30,
    n_splits=5,
    beta=2
)

gbdt6 = evaluate_gbdt_models(
    X6, slope6, g6,
    horizon_name="6m",
    preproc_tree=preproc_tree_6,
    top_frac=0.30,
    n_splits=5,
    beta=2
)

gbdt_results = pd.concat([gbdt3, gbdt6], ignore_index=True)

out_csv = os.path.join(OUT_TABLES, "step5_gbdt_results.csv")
gbdt_results.to_csv(out_csv, index=False)

print("Saved:", out_csv)
gbdt_results


In [ ]:
# só para ver rapidamente PR-AUC e F2 destes 2 modelos
g3 = gbdt_results[gbdt_results["horizon"]=="3m"]
g6 = gbdt_results[gbdt_results["horizon"]=="6m"]

plot_metric_bars(g3, "PR_AUC_mean", "PR_AUC_std",
                 "PR-AUC — LightGBM vs XGBoost (3m)",
                 outfile=os.path.join(OUT_FIGURES, "step5_pr_auc_gbdt_3m.png"),
                 baseline=0.30, y_min=0.25, y_max=0.50)

plot_metric_bars(g6, "PR_AUC_mean", "PR_AUC_std",
                 "PR-AUC — LightGBM vs XGBoost (6m)",
                 outfile=os.path.join(OUT_FIGURES, "step5_pr_auc_gbdt_6m.png"),
                 baseline=0.30, y_min=0.25, y_max=0.50)

plot_metric_bars(g3, "F2_mean", "F2_std",
                 "F2 — LightGBM vs XGBoost (3m)",
                 outfile=os.path.join(OUT_FIGURES, "step5_f2_gbdt_3m.png"),
                 baseline=None, y_min=None, y_max=None)

plot_metric_bars(g6, "F2_mean", "F2_std",
                 "F2 — LightGBM vs XGBoost (6m)",
                 outfile=os.path.join(OUT_FIGURES, "step5_f2_gbdt_6m.png"),
                 baseline=None, y_min=None, y_max=None)

In [ ]:
import os
import pandas as pd

lr_path   = os.path.join(OUT_TABLES, "step5_logreg_results.csv")
gbdt_path = os.path.join(OUT_TABLES, "step5_gbdt_results.csv")

lr = pd.read_csv(lr_path)
gbdt = pd.read_csv(gbdt_path)

# Normalise model name (for cleaner display)
lr["model"] = lr["model"].replace({"LogReg_balanced": "LogisticRegression"})

# Keep only the necessary columns
keep_cols = [
    "horizon", "model",
    "PR_AUC_mean", "PR_AUC_std",
    "F2_mean", "F2_std",
    "FP_mean", "FN_mean",
    "thr_decision_mean", "thr_decision_std"
]

lr_sub = lr[[c for c in keep_cols if c in lr.columns]].copy()
gbdt_sub = gbdt[[c for c in keep_cols if c in gbdt.columns]].copy()

combined = pd.concat([lr_sub, gbdt_sub], ignore_index=True)

# Sort for readability
combined = combined.sort_values(["horizon", "PR_AUC_mean"], ascending=[True, False])

# Save final table
out_csv = os.path.join(OUT_TABLES, "step5_compare_lr_lgbm_xgb.csv")
combined.to_csv(out_csv, index=False)

print("Saved:", out_csv)
combined


# Step 5.4.1 — Tuned XGBoost (Optuna) — Evaluation (6m)

In this section we load the best hyperparameters found by Optuna for XGBoost (6m) and evaluate with the same leakage-free protocol:
- GroupKFold (subject-wise)
- rapid/slow labels defined per fold (top 30% worst slopes on training)
- decision threshold chosen per fold to maximise F2
- metrics: PR-AUC, F2, FP/FN (mean ± std)


In [ ]:
# --- fix: ensure correct groups ---
# df6 must exist and have the subject_id column

g6 = df6["subject_id"].copy()
slope6 = df6["slope_180d_per_30d"].copy()

# If X6 already exists, verify consistency; otherwise recreate
drop_cols = {"subject_id", "slope_180d_per_30d"}
if "slope_90d_per_30d" in df6.columns:
    drop_cols.add("slope_90d_per_30d")

X6 = df6[[c for c in df6.columns if c not in drop_cols]].copy()

print("X6 rows:", len(X6))
print("g6 rows:", len(g6))
print("slope6 rows:", len(slope6))


In [ ]:
import os, json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, confusion_matrix

# 1) load best params
BEST_PATH = os.path.join(OUT_TABLES, "optuna_xgb_6m_best_params.json")
with open(BEST_PATH, "r", encoding="utf-8") as f:
    best = json.load(f)

xgb_params = best["best_params"].copy()

# add fixed parameters (same as in the Optuna script)
xgb_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
    "tree_method": "hist",
})

print("Loaded best PR-AUC:", best["best_value"])
print("Params:", xgb_params)

# 2) CV evaluation (uses X6/slope6/g6 and preproc_tree_6 already existing from Step 5)
def eval_xgb_tuned_6m(X, slope, groups, preproc_tree, params,
                     top_frac=0.30, n_splits=5, beta=2):

    gkf = GroupKFold(n_splits=n_splits)

    pr_list, f1_list, f2_list = [], [], []
    tn_list, fp_list, fn_list, tp_list = [], [], [], []
    thr_list = []

    for fold, (tr, te) in enumerate(gkf.split(X, groups=groups), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        slope_tr, slope_te = slope.iloc[tr], slope.iloc[te]

        thr_label = float(np.nanquantile(slope_tr.dropna(), top_frac))
        ytr = (slope_tr <= thr_label).astype(int)
        yte = (slope_te <= thr_label).astype(int)

        clf = XGBClassifier(**params)
        pipe = Pipeline([("prep", preproc_tree), ("clf", clf)])

        pipe.fit(Xtr, ytr)

        proba_tr = pipe.predict_proba(Xtr)[:, 1]
        thr_decision, _ = pick_threshold_max_fbeta(ytr.to_numpy(), proba_tr, beta=beta, grid=None)
        thr_list.append(float(thr_decision))

        proba_te = pipe.predict_proba(Xte)[:, 1]
        yhat = (proba_te >= thr_decision).astype(int)

        pr_list.append(average_precision_score(yte, proba_te))
        f1_list.append(f1_score(yte, yhat, zero_division=0))
        f2_list.append(fbeta_score(yte, yhat, beta=beta, zero_division=0))

        tn, fp, fn, tp = confusion_matrix(yte, yhat, labels=[0, 1]).ravel()
        tn_list.append(tn); fp_list.append(fp); fn_list.append(fn); tp_list.append(tp)

    return pd.DataFrame([{
        "horizon": "6m",
        "model": "XGBoost_tuned",
        "PR_AUC_mean": float(np.mean(pr_list)),
        "PR_AUC_std": float(np.std(pr_list)),
        "F1_mean": float(np.mean(f1_list)),
        "F1_std": float(np.std(f1_list)),
        "F2_mean": float(np.mean(f2_list)),
        "F2_std": float(np.std(f2_list)),
        "TN_mean": float(np.mean(tn_list)),
        "FP_mean": float(np.mean(fp_list)),
        "FN_mean": float(np.mean(fn_list)),
        "TP_mean": float(np.mean(tp_list)),
        "thr_decision_mean": float(np.mean(thr_list)),
        "thr_decision_std": float(np.std(thr_list)),
    }])

xgb_tuned_res6 = eval_xgb_tuned_6m(X6, slope6, g6, preproc_tree_6, xgb_params)
xgb_tuned_res6


In [ ]:
out_csv = os.path.join(OUT_TABLES, "step5_xgb_tuned_results_6m.csv")
xgb_tuned_res6.to_csv(out_csv, index=False)
print("Saved:", out_csv)
